# clock/10.1 — gaze heads vs concept-copier (dual-route induction) heads

Sibling of `10_goal_vs_FV_heads`, swapping the pair:
* **TARGET_SET = gaze heads** (our step-lookback heads) — cf. goal heads in nb10
* **probe set = concept-copier heads** (Feucht et al., dual-route induction) — cf. FV heads in nb10

1. Load both (top-K=200), plot locations + overlap.
2. **Sanity-check** the concept heads by replicating the dual-route demo (§2 of
   `Dual_Route_Model_of_Induction_Demo.ipynb`): on a *repeated* prompt they should attend from the
   2nd copy back into the 1st (the induction signature).
3. Do concept heads show **gaze** behavior? — the gaze visualization, fed the concept heads.

In [ ]:
%load_ext autoreload
%autoreload 2
import os, sys, json, glob, math, random, logging
sys.path.append("../..")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0,4")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import matplotlib.patches as mpatches
from rich.console import Console
from rich.theme import Theme
from rich.table import Table
from rich.markup import escape
from src.utils import logging_utils, env_utils
logging_utils.configure(level=logging.INFO)
logger = logging.getLogger(__name__)
console = Console(theme=Theme({"default": "white"}), highlight=False)
def rich_print(*a, **k): console.print(*a, **k)

In [ ]:
from src.models import ModelandTokenizer
MODEL_NAME = "Qwen/Qwen2.5-Coder-32B-Instruct"
mt = ModelandTokenizer(model_key=MODEL_NAME, dtype=torch.bfloat16,
                       device_map="auto", attn_implementation="eager")
N_LAYER, N_HEADS = mt.n_layer, mt.n_heads
print(f"n_layer={N_LAYER}  n_heads={N_HEADS}  n_embd={mt.n_embd}  head_dim={mt.head_dim}")

## 1. Head sets — gaze (TARGET) + concept-copier

The dual-route run writes `head_orderings/<model>/concept_copying.json` only **at completion**.
While it's still running we can rank heads from its **per-batch checkpoint** (the accumulators we
added): `score = patched.m1_prob/n − corrupt.m1_prob/n` — the same quantity the final JSON stores,
just over fewer examples (noisier, and skewed toward whichever n-gram buckets have run).

In [ ]:
TOP_K = 200
DR_ROOT = "../../existing_works/dual-route-induction"
_short = MODEL_NAME.split("/")[-1]

def load_copier_heads(scoretype="concept", topk=TOP_K):
    """Ranked (layer, head) list for the dual-route copier heads. Prefers the finished ordering
    JSON; falls back to the live per-batch checkpoint so we can look before the ~8-day run ends.

    NOTE the two scores are NOT symmetric (see convert_causal_scores.py, which produces the
    paper's head_orderings):
        concept = patched.m1 - corrupt.m1   (real entities;  m1 = 2nd entity token at -1)
        token   = patched.m2 - corrupt.m2   (random-token run; m2 = 1st entity token at -2)
    (causal_scores.py's own inline json uses m1 for both — the converter is the canonical one.)"""
    field = "m1_prob" if scoretype == "concept" else "m2_prob"
    j = f"{DR_ROOT}/cache/head_orderings/{_short}/{scoretype}_copying.json"
    if os.path.exists(j):
        ranked = [tuple(x) for x in json.load(open(j))]
        return ranked[:topk], f"{scoretype}_copying.json (final)", None
    ck = f"{DR_ROOT}/cache/causal_scores/{_short}/{scoretype}_len30_n1024.ckpt.pt"
    assert os.path.exists(ck), (
        f"neither {j} nor {ck} exists — finish/copy the dual-route run for {MODEL_NAME}")
    blob = torch.load(ck, map_location="cpu", weights_only=False)
    corrupt, patched = blob["corrupt"], blob["patched"]
    diff = patched[field] / patched["n"] - corrupt[field] / corrupt["n"]           # (L*H,)
    order = torch.topk(diff, k=len(diff)).indices
    ranked = [(int(i) // N_HEADS, int(i) % N_HEADS) for i in order]                # flatidx_to_grididx
    prov = (f"{scoretype} CHECKPOINT [{field}] — {blob['done_batches']} batches / "
            f"{corrupt['n']} examples (PARTIAL, noisy)")
    return ranked[:topk], prov, diff.reshape(N_LAYER, N_HEADS).numpy()

concept_heads, concept_prov, concept_grid = load_copier_heads("concept")
token_heads,   token_prov,   token_grid   = load_copier_heads("token")
print(f"concept-copier heads: {len(concept_heads)}  <- {concept_prov}")
print(f"token-copier   heads: {len(token_heads)}  <- {token_prov}")

from scripts.clock.score_condition_heads import load_head_scores
GAZE_QSPAN = 15
gaze_scores = load_head_scores(model=MODEL_NAME, version="conditions",
                               q_span=GAZE_QSPAN, compare_with="mean")["combined"]   # (L, H)
_flat = np.argsort(gaze_scores, axis=None)[::-1][:TOP_K]
gaze_heads = [(int(i // N_HEADS), int(i % N_HEADS)) for i in _flat]
print(f"gaze heads: {len(gaze_heads)}  (top-{TOP_K} step-lookback, q_span={GAZE_QSPAN})")
print("  top-10 concept:", concept_heads[:10])
print("  top-10 token  :", token_heads[:10])
print("  top-10 gaze   :", gaze_heads[:10])

### 1b. Location + overlap  *(same treatment as nb10 §3a)*

In [ ]:
cset, gset = set(concept_heads), set(gaze_heads)
inter, union = cset & gset, cset | gset
jacc = len(inter) / len(union) if union else 0.0

grid = np.zeros((N_LAYER, N_HEADS))
for (l, h) in gaze_heads:    grid[l, h] += 1        # 1 = gaze
for (l, h) in concept_heads: grid[l, h] += 2        # 2 = concept, 3 = both
cmap = ListedColormap(["#f0f0f0", "#55a868", "#8172b3", "#c44e52"])

fig, axes = plt.subplots(2, 1, figsize=(11, 8), height_ratios=[2, 1])
axes[0].imshow(grid.T, aspect="auto", cmap=cmap, vmin=0, vmax=3, origin="lower")
axes[0].set_xlabel("layer"); axes[0].set_ylabel("head")
axes[0].set_title(f"gaze vs concept-copier heads (top-{TOP_K} each)  ·  Jaccard={jacc:.3f}")
axes[0].legend(handles=[mpatches.Patch(color=c, label=l) for c, l in
                        [("#55a868", "gaze only"), ("#8172b3", "concept only"), ("#c44e52", "both")]],
               loc="upper left", fontsize=8)
# per-layer counts — where does each set live?
axes[1].plot(np.bincount([l for l, _ in gaze_heads], minlength=N_LAYER), color="#55a868", label="gaze")
axes[1].plot(np.bincount([l for l, _ in concept_heads], minlength=N_LAYER), color="#8172b3", label="concept")
axes[1].plot(np.bincount([l for l, _ in inter], minlength=N_LAYER), color="#c44e52", label="both")
axes[1].set_xlabel("layer"); axes[1].set_ylabel("# heads"); axes[1].grid(alpha=0.3); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

# overlap vs chance, and how it grows with K
t = Table(title=f"gaze vs concept-copier overlap  ({concept_prov})")
for c in ["K", "|inter|", "Jaccard", "expected by chance", "ratio"]:
    t.add_column(c, justify="right" if c != "K" else "left")
for K in [10, 25, 50, 100, TOP_K]:
    a, b = set(concept_heads[:K]), set(gaze_heads[:K])
    exp = K * K / (N_LAYER * N_HEADS)
    t.add_row(str(K), str(len(a & b)), f"{len(a & b) / len(a | b):.3f}", f"{exp:.1f}",
              f"{len(a & b) / exp:.2f}x" if exp else "—")
rich_print(t)
rich_print(f"[grey58]overlap heads (top-{TOP_K}): "
           f"{', '.join(f'L{l}H{h}' for l, h in sorted(inter)[:25])}"
           f"{' …' if len(inter) > 25 else ''}[/grey58]")

## 2. Sanity check — do these heads actually behave like concept copiers?

Replicates §2 of `Dual_Route_Model_of_Induction_Demo.ipynb` (`head_set_vis`): tokenize
`prompt + "\n" + prompt` and look at **value-weighted** attention from a token in the *second*
copy. Induction/copier heads attend back to the corresponding position in the *first* copy.

The demo's `get_l2_attn_weights` = softmax(attn) × ‖v‖, **row-renormalized**. Our
`get_attention_matrices(value_weighted=True)` computes the same product (and handles GQA via
`repeat_kv`, which the demo didn't need), so we only add the renormalization.

In [ ]:
from src.attention import get_attention_matrices

def l2_attn(text):
    """(layers, heads, q, k) value-weighted attention, row-renormalized — matches the demo."""
    A = np.asarray(get_attention_matrices(text, mt, value_weighted=True).attention_matrices)
    return A / np.clip(A.sum(axis=-1, keepdims=True), 1e-9, None)

DEMO_PROMPT = "By the false azure in the windowpane"      # the demo's sentence
FROMTOK, K_SHOW = -4, 8

rep = f"{DEMO_PROMPT}\n{DEMO_PROMPT}"
ai = get_attention_matrices(rep, mt, value_weighted=True)
toks = ai.tokenized_prompt
A = np.asarray(ai.attention_matrices)
A = A / np.clip(A.sum(axis=-1, keepdims=True), 1e-9, None)

def head_set_vis(heads, title, cmap="Purples"):
    g = np.stack([A[l, h, FROMTOK, :FROMTOK + 1] for (l, h) in heads[:K_SHOW]])
    fig, ax = plt.subplots(figsize=(min(0.42 * (len(toks) + FROMTOK + 1) + 2, 15), 3.2))
    im = ax.imshow(g, aspect="auto", cmap=cmap)
    ax.set_xticks(range(g.shape[1])); ax.set_xticklabels(toks[:FROMTOK + 1], rotation=90, fontsize=7)
    ax.set_yticks(range(len(heads[:K_SHOW])))
    ax.set_yticklabels([f"{l}.{h}" for l, h in heads[:K_SHOW]], fontsize=7)
    ax.set_title(f"{MODEL_NAME.split('/')[-1]}: Top-{K_SHOW} {title}   (attention FROM token {FROMTOK})",
                 fontsize=9)
    fig.colorbar(im, ax=ax, fraction=0.02); plt.tight_layout(); plt.show()

print(f"tokens ({len(toks)}):", toks)
head_set_vis(concept_heads, "Concept Induction Heads")
head_set_vis(gaze_heads, "Gaze Heads (contrast)", cmap="Greens")

### 2b. Quantitative smoke — first-copy attention mass

Over ~32 repeated prompts: how much of each head set's attention (from the last token) lands in the
**first copy**? Concept/induction heads should beat a random-head baseline; gaze heads are the
interesting middle case.

In [ ]:
subj = ["the cat", "a robin", "my friend", "the engineer", "her sister", "the old man",
        "a student", "the pilot"]
pred = ["walked to the river", "opened the wooden door", "counted the bright stars",
        "sold the red bicycle"]
SENTS = [f"{s} {p}" for s in subj for p in pred]          # 32 prompts

_rng = random.Random(0)
random_heads = [(_rng.randrange(N_LAYER), _rng.randrange(N_HEADS)) for _ in range(TOP_K)]

def first_copy_mass(head_sets, sents=SENTS):
    """{name: mean fraction of last-token attention landing in the 1st copy}."""
    acc = {k: [] for k in head_sets}
    for s in sents:
        A = l2_attn(f"{s}\n{s}")
        tk = get_attention_matrices(f"{s}\n{s}", mt, value_weighted=True).tokenized_prompt
        nl = next((i for i, t in enumerate(tk) if "\n" in t), len(tk) // 2)   # 1st copy = [0, nl)
        for name, hs in head_sets.items():
            rows = A[[l for l, _ in hs], [h for _, h in hs], -1, :]            # (n_heads, S)
            acc[name].append(rows[:, :nl].sum(axis=1).mean())
    return {k: float(np.mean(v)) for k, v in acc.items()}

mass = first_copy_mass({"concept": concept_heads, "gaze": gaze_heads, "random": random_heads})
t = Table(title=f"induction smoke — first-copy attention mass ({len(SENTS)} repeated prompts)")
t.add_column("head set"); t.add_column("mean 1st-copy attn", justify="right"); t.add_column("vs random", justify="right")
for k in ["concept", "gaze", "random"]:
    t.add_row(f"{k} (top-{TOP_K})", f"{mass[k]:.4f}",
              f"{mass[k] / max(mass['random'], 1e-9):.2f}x" if k != "random" else "—")
rich_print(t)
rich_print("[grey58]>1x for concept ⇒ they preferentially look back into the repeated copy "
           "= induction behavior (the sanity check we want before trusting the head set).[/grey58]")

## 3. Do concept-copier heads show *gaze* behavior?  *(qualitative)*

Feed the concept heads into our gaze visualization — per-step attention over the condition spans of
a `check_conditions` prompt (`track_conditions`). Gaze heads track *the condition currently being
implemented*; the question is whether concept copiers do anything similar.

In [ ]:
from src.attention import show_interactive_attention
from src.goals.multiline_prompts import build_example
from src.functional import generate_with_patch

demo_ex = build_example(mt=mt, rng=random.Random(42), target_slot=2, n_conditions=5,
                        mode="chat", naming="letters")
demo_prompt = demo_ex["full_prompt"]
demo_gen = generate_with_patch(mt, inputs=demo_prompt, max_new_tokens=128,
                               do_sample=False, remove_prefix=True, n_gen_per_prompt=1)[0]
rich_print(f"[white]{escape(demo_prompt)}[/white][sea_green1]{escape(demo_gen)}[/sea_green1]")

ai_g = get_attention_matrices(demo_prompt + demo_gen, mt, value_weighted=True)
tk = ai_g.tokenized_prompt
start = next((i - 1 for i, t in enumerate(tk) if "user" in t.replace("＜", "<")), 0)

K_GAZE = 50    # top-K of each set for a readable view
show_interactive_attention(attn_info=ai_g, selected_heads=sorted(concept_heads[:K_GAZE]),
                           start_from=max(0, start), agg="mean", normalize="row_max",
                           cmap="Purples", track_conditions=demo_ex["spec"], height=None,
                           title=f"CONCEPT-copier heads (top-{K_GAZE}) in the gaze view")

show_interactive_attention(attn_info=ai_g, selected_heads=sorted(gaze_heads[:K_GAZE]),
                           start_from=max(0, start), agg="mean", normalize="row_max",
                           cmap="Greens", track_conditions=demo_ex["spec"], height=None,
                           title=f"GAZE heads (top-{K_GAZE}) — reference")

### Next: systematic version

The qualitative view above is one prompt. We already have a pipeline that scores this over N
prompts (clean-vs-patched gaze, hard-flip / soft-Δ per head set) — **tell me which notebook to
lift it from** and I'll wire concept-vs-gaze through it at N=128.

Caveats for §1 while the dual-route run is unfinished: the ranking comes from a partial checkpoint
(fewer examples, n-gram-bucket skew), so treat head identities as provisional — rerun this cell
once `concept_copying.json` lands.

## 3b. Systematic: collective step-lookback (gaze) score  *(lifted from 09.1 §1b)*

Runs on **cached attention** (`load_cached` + `attn.npz`) — no model forward needed.

Per head, `score_one_sample` gives `attn(current condition) / mean_k attn(condition k)`; a ratio
**> 1** means the head preferentially attends to the condition being implemented *right now*. A head
set's **collective** score = the mean of that per-head ratio over the set.

Head sets (per spec): **goal** and **decision** load all their mask heads; **decision**,
**concept-copier** and **gaze** are then cut to the top 150; **random** is 150 draws.

In [ ]:
from pathlib import Path
from tqdm.auto import tqdm
from scripts.clock.score_condition_heads import load_cached, score_one_sample
from src.goals.oid_gaze import cache_root
from src.goals.head_loc_attn import HeadMask

VERSION, Q_SPAN, COMPARE = "conditions", 15, "mean"
N_SAMPLES = 256
TOPN = 150                      # gaze / decision / concept-copier / random
CACHE_ROOT = Path(cache_root(MODEL_NAME, VERSION))

# ── head sets ────────────────────────────────────────────────────────────────
# goal: ALL heads of the trained mask (threshold 0.9, band 25-55) — as in 12.1 / 13
GOAL_THRESHOLD, GOAL_BAND = 0.9, (25, 56)
_g = torch.load(os.path.join(env_utils.DEFAULT_RESULTS_DIR,
                f"localization/{_short}/head_mask.pt"), map_location="cpu")
_gl = [int(x) for x in _g["layers"]]
_gm = _g["mask_values"].numpy().copy(); _gm[:GOAL_BAND[0]] = 0; _gm[GOAL_BAND[1]:] = 0
goal_heads = [(_gl[li], int(h)) for li, h in np.argwhere(_gm > GOAL_THRESHOLD)]

# decision: load ALL mask values, then take the TOP-N by mask value (no threshold cut)
DEC_TAG = "lam10.0"
_root = os.path.join(cache_root(MODEL_NAME, VERSION), "completion_mask")
_cand = [t for t in os.listdir(_root) if os.path.isdir(os.path.join(_root, t)) and DEC_TAG in t]
assert _cand, f"no completion_mask with tag {DEC_TAG!r} under {_root}"
_latest = max(_cand, key=lambda t: os.path.getmtime(os.path.join(_root, t)))
_d = torch.load(os.path.join(_root, _latest, "mask.pt"), map_location="cpu", weights_only=False)
_dl = [int(x) for x in _d["layers"]]
_dv = _d["mask_values"].numpy()
_dord = np.argsort(_dv, axis=None)[::-1]                       # rank ALL heads by mask value
decision_heads = [(_dl[int(i) // N_HEADS], int(i) % N_HEADS) for i in _dord][:TOPN]

_rng0 = random.Random(0)
random_heads = list(zip(_rng0.choices(range(N_LAYER), k=TOPN), _rng0.choices(range(N_HEADS), k=TOPN)))

HEAD_SETS = {
    "goal (all)":              goal_heads,
    f"gaze (top-{TOPN})":      gaze_heads[:TOPN],
    f"decision (top-{TOPN})":  decision_heads,
    f"concept (top-{TOPN})":   concept_heads[:TOPN],
    f"token (top-{TOPN})":     token_heads[:TOPN],
    f"random ({TOPN})":        random_heads,
}
for k, v in HEAD_SETS.items():
    ls = sorted({l for l, _ in v})
    print(f"  {k:22s} n={len(v):4d}  layers {ls[0]}-{ls[-1]}")

# ── score over cached samples ────────────────────────────────────────────────
tasks = []
for mod in ["monolithic"]:
    mod_root, attn_root = CACHE_ROOT / mod, CACHE_ROOT / "attn" / mod
    if not mod_root.exists():
        continue
    for d in sorted(mod_root.iterdir()):
        ad = attn_root / d.name
        if (d / "labels.json").exists() and (d / "key_ranges.json").exists() and (ad / "attn.npz").exists():
            tasks.append((d.name, mod, d, ad))
tasks = tasks[:N_SAMPLES]
print(f"\nscoring {len(tasks)} cached samples (q_span={Q_SPAN}, compare_with={COMPARE})")

S = np.zeros((N_LAYER, N_HEADS)); Cn = np.zeros((N_LAYER, N_HEADS))
for name, mod, ld, ad in tqdm(tasks, desc="scoring samples"):
    labels, key_ranges, attn = load_cached(ld, ad)
    s, c = score_one_sample(attn, labels, key_ranges, mod, q_span=Q_SPAN, compare_with=COMPARE)
    S += s; Cn += c
per_head = np.divide(S, Cn, out=np.full((N_LAYER, N_HEADS), np.nan), where=Cn > 0)

def set_vals(hs):
    return np.array([per_head[l, h] for l, h in hs if not np.isnan(per_head[l, h])])

t = Table(title=f"collective step-lookback ratio (N={len(tasks)} cached samples; >1 = gaze behavior)")
for c in ["head set", "n", "mean", "median", "frac>1"]:
    t.add_column(c, justify=("left" if c == "head set" else "right"))
for name, hs in HEAD_SETS.items():
    v = set_vals(hs)
    t.add_row(name, str(len(v)), f"{v.mean():.3f}", f"{np.median(v):.3f}", f"{np.mean(v > 1):.2f}")
rich_print(t)

fig, ax = plt.subplots(figsize=(8.5, 4.2))
data = [set_vals(hs) for hs in HEAD_SETS.values()]
ax.boxplot(data, labels=[k.replace(" (", "\n(") for k in HEAD_SETS], showmeans=True, widths=0.6)
ax.axhline(1.0, ls="--", c="grey", lw=1, label="ratio = 1 (no gaze)")
ax.set_ylabel("per-head step-lookback ratio")
ax.set_title(f"do concept-copier heads gaze? (collective, N={len(tasks)})", fontsize=10)
ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3); plt.tight_layout(); plt.show()